[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/para_casa/S08_01_arboles_decision.ipynb)

# S08 · 01 — Árboles de decisión: sobreajuste, poda e importancia

Sprint 8 · M3 (UD 3.6)

**Contexto TurisData.** El gerente pide algo que **pueda leer sin ser matemático**: reglas del tipo «si la reserva es de tarifa flexible, sin depósito y con más de 100 días de antelación, el riesgo de cancelar es alto». Los **árboles de decisión** dan exactamente eso… siempre que no los dejemos crecer sin control.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)

# Los árboles NO necesitan escalar; solo imputar nulos y codificar las categóricas
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
nombres = prep.get_feature_names_out()
print("Columnas tras el preprocesado:", len(nombres))
print("Listo ✔")

### 👀 A2 · EJEMPLO — Un árbol pequeño, dibujado

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué pregunta hace el árbol primero (en la raíz)?
2. ¿Cuántas preguntas seguidas como máximo se hacen hasta llegar a una hoja?
3. ¿Qué significa que un recuadro sea más intenso de color?

In [ ]:
arbol_peq = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(Xtr, y_train)
plt.figure(figsize=(16, 6))
plot_tree(arbol_peq, feature_names=list(nombres), class_names=["se mantiene", "cancela"],
          filled=True, rounded=True, fontsize=8, proportion=True)
plt.title("Árbol de decisión de profundidad 3")
plt.show()

### 👀 A3 · EJEMPLO — Las reglas del árbol, en texto

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Cuál es la primera pregunta del árbol?
2. Una reserva con 200 días de antelación y de otro canal que no sea el directo, ¿qué predice?
3. ¿Cuántas hojas (líneas `class:`) tiene este árbol?

In [ ]:
print(export_text(arbol_peq, feature_names=list(nombres), max_depth=3))

### ✏️ A4 · EJERCICIO 1 de 4 — La variable de la raíz y el número de hojas

**Tu turno:** El árbol pequeño empieza preguntando por la variable que mejor separa. Guarda en `variable_raiz` el **nombre** de esa variable (pista: `arbol_peq.tree_.feature[0]` es el índice de la columna en `nombres`) y en `n_hojas` cuántas hojas tiene (`arbol_peq.get_n_leaves()`).

**Debe salir:** `El árbol empieza preguntando por: num__antelacion_dias` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué valen `variable_raiz` y `n_hojas`?
2. ¿Por qué una profundidad 3 da como máximo 8 hojas?
3. ¿Por qué la raíz pregunta por la antelación y no por otra variable?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: nombres[arbol_peq.tree_.feature[0]] y arbol_peq.get_n_leaves()
variable_raiz = ...
n_hojas = ...

assert isinstance(variable_raiz, str) and len(variable_raiz) > 3, "variable_raiz debe ser el nombre de una columna"
assert 2 <= n_hojas <= 8, "Un árbol de profundidad 3 tiene como máximo 8 hojas"
print("El árbol empieza preguntando por:", variable_raiz)
print("✔ Ejercicio 1 correcto")

### ✏️ A5 · EJERCICIO 2 de 4 — El árbol libre memoriza

**Tu turno:** Entrena `arbol_libre = DecisionTreeClassifier(random_state=SEMILLA)` (sin límites) con train. Guarda `auc_train_libre` y `auc_test_libre` (AUC con `predict_proba`) y `hojas_libre` (número de hojas).

**Debe salir:** `Árbol libre: 1206 hojas · AUC train 1.000 · AUC test 0.587` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué AUC saca el árbol libre en train y en test?
2. ¿Cuántas hojas tiene y qué te dice?
3. ¿Qué indica una diferencia tan grande entre train (1.000) y test (0.587)?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: roc_auc_score(y, modelo.predict_proba(X)[:, 1]) en train y en test
arbol_libre = ...
auc_train_libre = ...
auc_test_libre = ...
hojas_libre = ...

assert auc_train_libre > 0.99, "El árbol libre memoriza el entrenamiento: AUC ≈ 1"
assert auc_test_libre < 0.70, "…pero en test cae mucho (< 0,70): sobreajuste"
assert hojas_libre > 500, "Un árbol libre tiene cientos de hojas"
print(f"Árbol libre: {hojas_libre} hojas · AUC train {auc_train_libre:.3f} · AUC test {auc_test_libre:.3f}")
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Cómo cambia el AUC al podar con min_samples_leaf

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué AUC en validación cruzada tiene `min_samples_leaf=1` y cuál el mejor valor?
2. ¿Por qué baja el AUC cuando `min_samples_leaf` pasa de 160 a 320?
3. ¿Por qué se usa validación cruzada sobre train y no el test?

In [ ]:
hojas_min = [1, 5, 10, 20, 40, 80, 160, 320]
auc_cv_hojas = []
for m in hojas_min:
    arb = Pipeline([("prep", prep), ("arbol", DecisionTreeClassifier(min_samples_leaf=m, random_state=SEMILLA))])
    auc_cv_hojas.append(cross_val_score(arb, X_train, y_train, cv=cv, scoring="roc_auc").mean())
plt.figure(figsize=(7, 3.8))
plt.plot(hojas_min, auc_cv_hojas, marker="o", color="#2a6f97")
plt.xscale("log")
plt.title("Poda: AUC (validación cruzada) según el mínimo de reservas por hoja")
plt.xlabel("min_samples_leaf (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.show()

### ✏️ A7 · EJERCICIO 3 de 4 — Elegir la poda y medirla en test

**Tu turno:** Con las listas `hojas_min` y `auc_cv_hojas`, guarda en `mejor_min_hoja` el valor con mejor AUC. Entrena `arbol_podado` con ese valor sobre train y guarda `auc_test_podado` (AUC en test).

**Debe salir:** `min_samples_leaf = 80 · AUC test: 0.716 (libre: 0.587) · hojas: 46` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué `mejor_min_hoja` sale y qué AUC da en test?
2. ¿Cuánto mejora el árbol podado al libre en test y con cuántas hojas?
3. ¿Qué ganas y qué pierdes con la poda?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: np.argmax(auc_cv_hojas) da la posición del mejor; después DecisionTreeClassifier(min_samples_leaf=...)
mejor_min_hoja = ...
arbol_podado = ...
auc_test_podado = ...

assert mejor_min_hoja >= 10, "Con estos datos ruidosos, hojas de al menos 10 reservas funcionan mejor"
assert auc_test_podado > auc_test_libre + 0.08, "El árbol podado debería ganar claramente al libre en test"
print(f"min_samples_leaf = {mejor_min_hoja} · AUC test: {auc_test_podado:.3f} (libre: {auc_test_libre:.3f}) · hojas: {arbol_podado.get_n_leaves()}")
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Importancia de variables del árbol podado

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué variable es la más importante y cuánta importancia tiene?
2. ¿Coincide con lo que vimos en la raíz del árbol pequeño?
3. ¿Por qué no se puede decir «la antelación **causa** las cancelaciones» solo con este gráfico?

In [ ]:
importancias = pd.Series(arbol_podado.feature_importances_, index=nombres).sort_values(ascending=False)
plt.figure(figsize=(7, 4.5))
importancias.head(10).iloc[::-1].plot(kind="barh", color="#2a6f97")
plt.title("Importancia de variables del árbol podado (top 10)")
plt.xlabel("Importancia (reducción de impureza)")
plt.ylabel("")
plt.tight_layout()
plt.show()

### ✏️ A9 · EJERCICIO 4 de 4 — Las tres variables más importantes

**Tu turno:** Guarda en `top3` la lista de los **nombres** de las 3 variables más importantes de `arbol_podado` (usa la Series `importancias` ya ordenada) y en `suma_top3` cuánta importancia acumulan entre las tres.

**Debe salir:** `Top 3: ['num__antelacion_dias', 'cat__deposito_sin_deposito', 'cat__canal_directo'] · acumulan 70% de la importancia` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuáles son las tres variables y cuánto suman?
2. ¿Qué te dice que solo 3 columnas de 34 acumulen el 70 %?
3. ¿Cuál de las tres es una variable numérica y cuáles categóricas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: importancias.head(3).index y .sum()
top3 = ...
suma_top3 = ...

assert len(top3) == 3 and all(isinstance(n, str) for n in top3), "top3 debe ser una lista de 3 nombres"
assert 0.3 < suma_top3 <= 1, "Las tres primeras deberían acumular una parte importante"
print("Top 3:", top3, f"· acumulan {suma_top3:.0%} de la importancia")
print("✔ Ejercicio 4 correcto")

### 👀 A10 · EJEMPLO — Cinco árboles, cinco submuestras: ¿coinciden?

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿En qué porcentaje de las reservas de test no coinciden los 5 árboles?
2. ¿Qué quiere decir en la práctica?
3. ¿Qué técnicas arreglan esta inestabilidad?

In [ ]:
predicciones = []
for semilla in range(5):
    idx = np.random.default_rng(semilla).choice(len(Xtr), int(0.8 * len(Xtr)), replace=False)
    a = DecisionTreeClassifier(min_samples_leaf=10, random_state=SEMILLA).fit(Xtr[idx], y_train.to_numpy()[idx])
    predicciones.append(a.predict(Xte))
predicciones = np.array(predicciones)
# % de reservas de test en las que los 5 árboles NO se ponen de acuerdo
desacuerdo = (predicciones.min(axis=0) != predicciones.max(axis=0)).mean()
print(f"En el {desacuerdo:.0%} de las reservas de test, los 5 árboles no coinciden en la predicción.")

### ✏️ A11 · MINI-RETO — Reglas para el gerente

**Tu turno:** Entrena `arbol_gerente = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEMILLA)` con train. Después escribe en la lista `reglas` **tres frases** en lenguaje de negocio (con cifras si puedes) que resuman lo que hace el árbol (mira `export_text`). Guarda su AUC en test en `auc_test_gerente`.

**Debe salir:** ✔ Mini-reto correcto y el árbol impreso con `export_text` (AUC en test ≈ 0,69)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Cuál es el AUC en test del árbol del gerente?
2. ¿Por qué es peor que el árbol podado (0,716)?
3. ¿Qué ventaja tiene para el gerente un árbol de 8 hojas frente a uno de 46?

In [ ]:
# pista: entrena el árbol, imprime export_text y redacta las reglas a partir de lo que veas
arbol_gerente = ...
auc_test_gerente = ...
print(export_text(arbol_gerente, feature_names=list(nombres)))
reglas = [...]

assert 0.6 < auc_test_gerente < 0.85, "El AUC del árbol de profundidad 3 debería estar entre 0,6 y 0,85"
assert len(reglas) == 3 and all(isinstance(r, str) and len(r) > 20 for r in reglas), "Escribe tres frases completas"
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, el árbol de decisión, sus utilidades de dibujo y texto y el preprocesado.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv` (6000 reservas), crea `temporada_alta` y define `X` (9 numéricas y 6 categóricas) e `y` (si la reserva se cancela).
    - Reparte train y test (80 % / 20 %, estratificado: 4800 y 1200 reservas) y define `cv`, la validación cruzada de 5 folds.
    - Define `prep`: los árboles **no necesitan escalar**, solo imputar los nulos y pasar las categóricas a *one-hot*. Produce `Xtr` y `Xte` y los nombres de columna en `nombres`.
- **Qué debe salir:**
    - `Columnas tras el preprocesado: 34` y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea: preguntas de sí/no encadenadas

#### A2 · Ejemplo — Un árbol pequeño, dibujado

Un árbol de decisión busca, entre todas las variables y umbrales posibles, **la pregunta que mejor separa** las reservas canceladas de las no canceladas (medido con el índice de **Gini** o la entropía: cuanto más «pura» la separación, mejor). Luego repite el proceso en cada rama hasta que se cumple una condición de parada. Cada **hoja** final da una predicción (la clase mayoritaria en ella). Lo vemos con un árbol **pequeño** (profundidad 3).

- **Qué hace:**
    - `DecisionTreeClassifier(max_depth=3, ...)` crea un árbol que solo puede encadenar 3 preguntas; `.fit(Xtr, y_train)` lo construye con las reservas de train.
    - `plot_tree(...)` lo dibuja: cada recuadro es una pregunta (o una hoja), con su porcentaje de reservas (`proportion=True`) y su color según la clase mayoritaria (`filled=True`).
- **Qué debe salir:**
    - Un dibujo de un árbol de 3 niveles de preguntas, con 8 hojas en la fila de abajo. Arriba, la pregunta por la antelación; debajo, preguntas sobre el depósito y el canal.
- **Cómo leerlo:**
    - Cada recuadro dice: la pregunta (si es cierta, ve a la izquierda; si no, a la derecha), el `gini` (impureza: 0 = todas de la misma clase), el porcentaje de reservas y la clase mayoritaria. Cuanto más intenso el color, más pura la hoja.
- **Si falla:**
    - `NameError: name 'Xtr' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué pregunta hace el árbol primero (en la raíz)?* → Por la **antelación** (`antelacion_dias`): según el texto del ejemplo siguiente, `antelacion_dias <= 110.5`.
2. *¿Cuántas preguntas seguidas como máximo se hacen hasta llegar a una hoja?* → **3**, por `max_depth=3`.
3. *¿Qué significa que un recuadro sea más intenso de color?* → Que la **hoja es más pura**: casi todas sus reservas son de la misma clase.

</details>

#### A3 · Ejemplo — Las reglas del árbol, en texto

- **Qué hace:**
    - `export_text(arbol_peq, feature_names=...)` escribe el árbol como reglas anidadas: cada línea `|---` es una pregunta y cada `class:` es una hoja con su predicción (0 = se mantiene, 1 = cancela).
- **Qué debe salir:**
    - El árbol en texto: la raíz es `num__antelacion_dias <= 110.50`. Con ≤ 110 días: si hay depósito (`deposito_sin_deposito <= 0.50`) → clase 0; si no hay depósito: clase 1 solo si la reserva viene de `canal_web_intermediaria`, y clase 0 en caso contrario. Con > 110 días: clase 1 salvo si es del canal directo y la antelación es ≤ 163,5 días (clase 0).
- **Cómo leerlo:**
    - Se lee de arriba abajo, siguiendo la rama cuya pregunta se cumple. Las variables aparecen con prefijo (`num__` si era numérica, `cat__variable_valor` si venía de una categórica: 1 = sí y 0 = no).
- **Si falla:**
    - `NameError: name 'arbol_peq' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la primera pregunta del árbol?* → `antelacion_dias <= 110.5`: ¿se reservó con 110 días o menos de antelación?
2. *Una reserva con 200 días de antelación y de otro canal que no sea el directo, ¿qué predice?* → **Cancela** (clase 1): sale por `antelacion > 110.5` y `canal_directo <= 0.5`.
3. *¿Cuántas hojas (líneas `class:`) tiene este árbol?* → **8** (el máximo para profundidad 3, 2³).

</details>

#### A4 · Ejercicio 1 — La variable de la raíz y el número de hojas

- **Qué hace:**
    - `arbol_peq.tree_.feature[0]` es el **índice** de la columna por la que pregunta la raíz (el nodo 0).
    - `nombres[...]` traduce ese índice al nombre de la columna (`variable_raiz`).
    - `arbol_peq.get_n_leaves()` cuenta las hojas (`n_hojas`).
- **Qué debe salir:**
    - `El árbol empieza preguntando por: num__antelacion_dias`.
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `variable_raiz debe ser el nombre de una columna` → has guardado el índice (un número) y no el nombre: busca ese índice en `nombres[...]`.
    - `Un árbol de profundidad 3 tiene como máximo 8 hojas` → `n_hojas` no es `arbol_peq.get_n_leaves()`.
    - `NameError: name 'arbol_peq' is not defined` → falta ejecutar los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `variable_raiz` y `n_hojas`?* → `'num__antelacion_dias'` y **8**.
2. *¿Por qué una profundidad 3 da como máximo 8 hojas?* → Porque cada pregunta **duplica** las ramas: 2 × 2 × 2 = 2³ = 8.
3. *¿Por qué la raíz pregunta por la antelación y no por otra variable?* → Porque, de todas las preguntas posibles, es la que **mejor separa** canceladas y no canceladas (menor impureza de Gini). Es la variable que más información da.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
variable_raiz = nombres[arbol_peq.tree_.feature[0]]
n_hojas = arbol_peq.get_n_leaves()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Sobreajuste: el árbol libre

#### A5 · Ejercicio 2 — El árbol libre memoriza

Si no le ponemos límite, el árbol sigue preguntando hasta que **cada hoja es pura**, es decir, **memoriza** el entrenamiento (incluido el ruido).

- **Qué hace:**
    - `DecisionTreeClassifier(random_state=SEMILLA)` crea un árbol sin límites y `.fit(Xtr, y_train)` lo construye.
    - `roc_auc_score(y, modelo.predict_proba(X)[:, 1])` mide el AUC con la probabilidad de cancelar; se calcula en train y en test.
    - `get_n_leaves()` cuenta las hojas.
- **Qué debe salir:**
    - `Árbol libre: 1206 hojas · AUC train 1.000 · AUC test 0.587`.
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `El árbol libre memoriza el entrenamiento: AUC ≈ 1` → has puesto algún límite (`max_depth`…) o has medido el AUC con `predict` y no con `predict_proba`.
    - `…pero en test cae mucho (< 0,70): sobreajuste` → has medido en test con el árbol equivocado, o has entrenado con test.
    - `NameError: name 'Xte' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC saca el árbol libre en train y en test?* → **1.000** en train y **0.587** en test.
2. *¿Cuántas hojas tiene y qué te dice?* → **1206 hojas** para solo 4800 reservas: casi **una hoja por cada 4 reservas**. Ha memorizado el entrenamiento.
3. *¿Qué indica una diferencia tan grande entre train (1.000) y test (0.587)?* → **Sobreajuste**: el árbol aprende el ruido del entrenamiento y no generaliza; en test casi no supera al azar (0,5).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
arbol_libre = DecisionTreeClassifier(random_state=SEMILLA).fit(Xtr, y_train)
auc_train_libre = roc_auc_score(y_train, arbol_libre.predict_proba(Xtr)[:, 1])
auc_test_libre = roc_auc_score(y_test, arbol_libre.predict_proba(Xte)[:, 1])
hojas_libre = arbol_libre.get_n_leaves()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Poda: frenar el crecimiento

#### A6 · Ejemplo — Cómo cambia el AUC al podar con min_samples_leaf

La **poda** limita la complejidad. Tres formas habituales:
- `max_depth`: profundidad máxima (nº de preguntas seguidas).
- `min_samples_leaf`: mínimo de reservas por hoja (evita hojas «de 2 casos»).
- `ccp_alpha`: poda **a posteriori**: se construye el árbol y se recortan las ramas que no compensan su complejidad (parámetro de coste-complejidad).

Miramos cómo cambia el AUC de **validación cruzada** (nunca el de test) con `min_samples_leaf`.

- **Qué hace:**
    - `hojas_min` es la lista de valores de `min_samples_leaf` a probar (de 1 a 320).
    - Para cada uno, `cross_val_score(..., scoring="roc_auc")` calcula el AUC medio en 5 folds sobre **train** (con el preprocesado dentro del pipeline) y lo guarda en `auc_cv_hojas`.
    - Dibuja el AUC frente a `min_samples_leaf` (eje logarítmico). Tarda unos segundos.
- **Qué debe salir:**
    - Un gráfico: el AUC sube de ≈ 0,58 (con 1) hasta ≈ 0,72 (con 80) y vuelve a bajar a ≈ 0,69 (con 320). `auc_cv_hojas` ≈ `[0.584, 0.635, 0.661, 0.686, 0.709, 0.716, 0.715, 0.692]`.
- **Cómo leerlo:**
    - Forma de «colina»: a la izquierda, árboles demasiado libres (sobreajuste); a la derecha, demasiado podados (subajuste). El máximo está en medio (80–160).
- **Si falla:**
    - `NameError: name 'prep' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC en validación cruzada tiene `min_samples_leaf=1` y cuál el mejor valor?* → **0,584** con 1 hoja mínima y el mejor, **0,716**, con `min_samples_leaf=80` (con 160 casi igual: 0,715).
2. *¿Por qué baja el AUC cuando `min_samples_leaf` pasa de 160 a 320?* → Porque el árbol se vuelve **demasiado simple** (subajuste): con hojas de al menos 320 reservas ya no puede captar patrones útiles.
3. *¿Por qué se usa validación cruzada sobre train y no el test?* → Porque el test debe quedar **sin tocar** hasta el final; si eligiéramos el parámetro mirándolo, ya no nos daría una estimación honesta.

</details>

#### A7 · Ejercicio 3 — Elegir la poda y medirla en test

- **Qué hace:**
    - `np.argmax(auc_cv_hojas)` da la **posición** del mayor AUC; `hojas_min[...]` el valor de `min_samples_leaf` de esa posición (`mejor_min_hoja`).
    - `DecisionTreeClassifier(min_samples_leaf=mejor_min_hoja, ...)` con `.fit(Xtr, y_train)` crea el árbol podado.
    - `roc_auc_score(y_test, ...)` mide su AUC en test (`auc_test_podado`); el `print` lo compara con el del árbol libre.
- **Qué debe salir:**
    - `min_samples_leaf = 80 · AUC test: 0.716 (libre: 0.587) · hojas: 46`.
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `Con estos datos ruidosos, hojas de al menos 10 reservas funcionan mejor` → `mejor_min_hoja` no sale de `auc_cv_hojas` (¿has cogido `max(auc_cv_hojas)`, que es el AUC, en vez de la posición con `argmax`?).
    - `El árbol podado debería ganar claramente al libre en test` → has entrenado el podado con test o no has usado `mejor_min_hoja`.
    - `NameError: name 'auc_cv_hojas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué `mejor_min_hoja` sale y qué AUC da en test?* → **80** y AUC test **0.716**.
2. *¿Cuánto mejora el árbol podado al libre en test y con cuántas hojas?* → De **0.587 a 0.716** (+0,13) y pasa de **1206 a 46 hojas** (≈ 26 veces menos).
3. *¿Qué ganas y qué pierdes con la poda?* → Ganas **generalización** (mejor AUC en test) y un árbol mucho más simple; pierdes el ajuste perfecto en train (que de todas formas era memorización).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
mejor_min_hoja = hojas_min[int(np.argmax(auc_cv_hojas))]
arbol_podado = DecisionTreeClassifier(min_samples_leaf=mejor_min_hoja, random_state=SEMILLA).fit(Xtr, y_train)
auc_test_podado = roc_auc_score(y_test, arbol_podado.predict_proba(Xte)[:, 1])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Importancia de variables

#### A8 · Ejemplo — Importancia de variables del árbol podado

El árbol calcula cuánto **reduce la impureza** (Gini) cada variable en total: es la `feature_importances_` (suma 1). Ayuda a explicar *qué mira* el modelo. Ojo:
- es una importancia **del modelo**, no una relación causal;
- tiende a inflar las variables numéricas con muchos valores distintos;
- en variables correlacionadas se reparte de forma algo arbitraria.

- **Qué hace:**
    - `arbol_podado.feature_importances_` da un valor por columna; `pd.Series(..., index=nombres).sort_values(ascending=False)` las ordena de mayor a menor (`importancias`).
    - Dibuja las 10 primeras en un gráfico de barras horizontales (`.iloc[::-1]` las invierte para que la mayor quede arriba).
- **Qué debe salir:**
    - Un gráfico de barras: arriba `num__antelacion_dias` (≈ 0,45), luego `cat__deposito_sin_deposito` (≈ 0,16), `cat__canal_directo` (≈ 0,09), `cat__canal_web_intermediaria` (≈ 0,08) y `cat__tarifa_flexible` (≈ 0,08).
- **Si falla:**
    - `NameError: name 'arbol_podado' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué variable es la más importante y cuánta importancia tiene?* → **`antelacion_dias`**, con ≈ **0,45**.
2. *¿Coincide con lo que vimos en la raíz del árbol pequeño?* → **Sí**: la raíz preguntaba por la antelación; la variable que más separa es también la más importante.
3. *¿Por qué no se puede decir «la antelación **causa** las cancelaciones» solo con este gráfico?* → Porque la importancia mide **cuánto usa el modelo** una variable, no una relación causal: puede haber variables relacionadas con ella que se «reparten» la importancia.

</details>

#### A9 · Ejercicio 4 — Las tres variables más importantes

- **Qué hace:**
    - `importancias.head(3)` toma las 3 primeras (la Series ya está ordenada); `.index` son sus nombres y `list(...)` los pasa a lista (`top3`).
    - `importancias.head(3).sum()` suma su importancia (`suma_top3`); `float(...)` lo pasa a número normal.
- **Qué debe salir:**
    - `Top 3: ['num__antelacion_dias', 'cat__deposito_sin_deposito', 'cat__canal_directo'] · acumulan 70% de la importancia`.
    - `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `top3 debe ser una lista de 3 nombres` → has guardado los valores y no los nombres: usa `.index`.
    - `Las tres primeras deberían acumular una parte importante` → has sumado otra cosa (¿`importancias.sum()`, que vale 1?).
    - `NameError: name 'importancias' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuáles son las tres variables y cuánto suman?* → `antelacion_dias`, `deposito_sin_deposito` y `canal_directo`: **≈ 70 %** (0,445 + 0,163 + 0,092).
2. *¿Qué te dice que solo 3 columnas de 34 acumulen el 70 %?* → Que el árbol se apoya en **muy pocas variables**: antelación, si hay depósito y si es canal directo. El resto aporta poco.
3. *¿Cuál de las tres es una variable numérica y cuáles categóricas?* → `antelacion_dias` es **numérica** (prefijo `num__`); las otras dos vienen de categóricas pasadas a *one-hot* (prefijo `cat__`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
top3 = list(importancias.head(3).index)
suma_top3 = float(importancias.head(3).sum())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Inestabilidad: el gran defecto

#### A10 · Ejemplo — Cinco árboles, cinco submuestras: ¿coinciden?

Un árbol cambia si cambian un poco los datos. Lo comprobamos entrenando 5 árboles (iguales en configuración) con distintas submuestras del 80 % de train y comparando **qué predicen para las mismas reservas de test**.

- **Qué hace:**
    - Para cada semilla (0 a 4), elige al azar el 80 % de las reservas de train **sin repetir** (`replace=False`), entrena un árbol con `min_samples_leaf=10` y guarda sus predicciones sobre test.
    - `predicciones.min(axis=0) != predicciones.max(axis=0)` marca las reservas de test en las que **no todos** los árboles dicen lo mismo; `.mean()` es la proporción.
    - El `print` la da en porcentaje.
- **Qué debe salir:**
    - `En el 57% de las reservas de test, los 5 árboles no coinciden en la predicción.`
- **Cómo leerlo:**
    - Este defecto es el que arreglan los *ensembles* (bosques aleatorios y boosting) que verás a continuación.
- **Si falla:**
    - `NameError: name 'Xtr' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué porcentaje de las reservas de test no coinciden los 5 árboles?* → En el **57 %**.
2. *¿Qué quiere decir en la práctica?* → Que **más de la mitad de las predicciones depende de qué 80 % de datos se usó**: el árbol es **inestable**. Si cambia un poco el entrenamiento, cambia su opinión.
3. *¿Qué técnicas arreglan esta inestabilidad?* → Los *ensembles*: **bosques aleatorios** (promedian muchos árboles) y **boosting**. Se ven en los dos cuadernos siguientes.

</details>

## 6. Mini-reto: reglas para el gerente

#### A11 · Mini-reto — Reglas para el gerente

- **Qué hace:**
    - `DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, ...)` con `.fit(Xtr, y_train)` crea un árbol pequeño y legible (8 hojas).
    - `roc_auc_score(y_test, ...)` mide su AUC en test (`auc_test_gerente`).
    - `export_text` lo imprime en reglas y `reglas` es la lista de tres frases en lenguaje de negocio (tuya, no tiene que coincidir con la de referencia).
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y el árbol en texto (el mismo de antes: raíz en `antelacion_dias <= 110.50`). `auc_test_gerente` ≈ 0,69.
- **Cómo leerlo:**
    - Tus tres reglas deben salir de las ramas del árbol: antelación ≤ 110 días, presencia de depósito, canal y antelación > 110 días. Puedes citar cifras de antelación y de canal.
- **Si falla:**
    - `El AUC del árbol de profundidad 3 debería estar entre 0,6 y 0,85` → `auc_test_gerente` no es el AUC en test de un árbol de profundidad 3 (¿lo has medido en train?).
    - `Escribe tres frases completas` → `reglas` debe ser una lista de exactamente 3 textos de más de 20 caracteres.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es el AUC en test del árbol del gerente?* → **≈ 0,69** (0,692).
2. *¿Por qué es peor que el árbol podado (0,716)?* → Porque es **más simple**: 8 hojas (frente a 46), así que capta menos matices. A cambio, se puede leer entero en 8 reglas.
3. *¿Qué ventaja tiene para el gerente un árbol de 8 hojas frente a uno de 46?* → Que **puede leerlo y explicarlo**: son 8 reglas con las variables que importan, sin necesidad de matemáticas.

</details>

## 7. Para reflexionar

1. El árbol libre acierta el 100 % en train. ¿Por qué es una mala noticia?
2. ¿Qué pierdes al limitar la profundidad y qué ganas?
3. ¿Por qué los árboles no necesitan escalar las variables?
4. Si el gerente pide «por qué se marcó esta reserva concreta como de riesgo», ¿qué ventaja tiene un árbol frente a una red neuronal?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque acertar el 100 % en train significa que **ha memorizado** el entrenamiento, ruido incluido. En test (reservas nuevas) cae a un AUC de 0,587, casi azar. Es sobreajuste.
2. Pierdes el ajuste fino al entrenamiento (el árbol ya no es «perfecto» en train) y algo de detalle. Ganas **generalización** (el AUC en test pasa de 0,587 a 0,716), un árbol mucho más pequeño (46 hojas frente a 1206) y que se pueda leer.
3. Porque un árbol solo hace preguntas de la forma «¿variable ≤ umbral?». Escalar no cambia el **orden** de los valores, así que las mejores preguntas son las mismas con o sin escalado.
4. El árbol permite **seguir el camino de esa reserva** por las preguntas (por ejemplo: antelación > 110, no es canal directo → cancela) y dar una regla concreta. Una red neuronal tiene miles de pesos y no ofrece una explicación directa.

</details>

## 8. Resumen: qué has aprendido
- Un árbol = preguntas de sí/no encadenadas que maximizan la pureza de cada rama.
- Sin límites **memoriza**; la poda (`max_depth`, `min_samples_leaf`, `ccp_alpha`) controla el sobreajuste.
- Es muy **explicable** y no necesita escalado, pero es **inestable** y suele rendir menos que los ensembles.